# Análise de Doenças Cardíacas - Versão para Google Colab

**Observação:** A primeira célula de código abaixo instala as bibliotecas necessárias para executar esta análise no ambiente do Google Colab.

In [ ]:
!pip install ucimlrepo pandas numpy scikit-learn matplotlib seaborn

## 1. Introdução

**Objetivo:** Este trabalho tem como objetivo aplicar técnicas de aprendizado de máquina supervisionado (Classificação) e não supervisionado (Clusterização) em um banco de dados público, seguindo as diretrizes da avaliação formativa de Inteligência Artificial.

**Dataset:** O conjunto de dados escolhido foi o "Heart Disease" do repositório da UCI, que contém informações clínicas de pacientes para avaliar a presença de doenças cardíacas.

**Tarefas:**
1.  **Classificação:** Construir modelos para prever se um paciente possui ou não doença cardíaca (diagnóstico binário).
2.  **Clusterização:** Agrupar pacientes com perfis clínicos semelhantes, o que pode ajudar a identificar diferentes grupos de risco ou características comuns.

## 2. Carregamento e Descrição dos Dados

**Objetivo:** A primeira etapa consiste em carregar as bibliotecas essenciais para a análise e importar o dataset. Utilizaremos a biblioteca `ucimlrepo` para facilitar o acesso direto aos dados da UCI.

In [ ]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

heart_disease = fetch_ucirepo(id=45)
X = heart_disease.data.features
y = heart_disease.data.targets
y_binary = (y > 0).astype(int)

print("--- Visão Geral dos Dados ---")
print(f"O dataset possui {X.shape[0]} amostras e {X.shape[1]} atributos.")
print("\nAtributos (colunas):")
print(list(X.columns))
print("\nPrimeiras 5 amostras:")
display(X.head())

## 3. Análise Exploratória de Dados (EDA) e Pré-Processamento

**Objetivo:** Antes de treinar os modelos, é crucial entender a natureza dos dados. Nesta etapa, vamos:
1.  Verificar a integridade dos dados (valores ausentes, tipos de dados).
2.  Realizar o tratamento necessário (imputação de dados faltantes).
3.  Visualizar a distribuição das variáveis e suas correlações para extrair insights.

### 3.1. Verificação de Dados Ausentes e Tipos

In [ ]:
print("--- Verificação de Tipos e Valores Nulos ---")
X.info()

**Análise:** A saída acima confirma que as colunas `ca` (número de vasos principais coloridos por fluoroscopia) e `thal` (resultado do teste de tálio) possuem valores ausentes. Todos os outros atributos estão completos.

### 3.2. Tratamento de Valores Nulos

**Justificativa:** Como a quantidade de dados ausentes é pequena, remover as linhas poderia levar à perda de informação valiosa. Uma abordagem mais segura é a **imputação**. Para variáveis como `ca` e `thal`, que são categóricas representadas por números, a melhor estratégia é preencher os valores faltantes com a **moda** (o valor mais frequente), pois isso preserva a distribuição original da variável.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='most_frequent')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

for col in X.columns:
    if X[col].dtype == 'float64':
        X_imputed[col] = X_imputed[col].astype(float)
    else:
        X_imputed[col] = X_imputed[col].astype(int)

print("--- Dados após a imputação ---")
X_imputed.info()

### 3.3. Análise Visual dos Dados

**Objetivo:** Gerar gráficos para entender a distribuição, o balanceamento e as correlações entre as variáveis.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

plt.rcParams['figure.figsize'] = (20, 15)
X_imputed.hist(bins=20)
plt.suptitle('Distribuição das Variáveis Numéricas', size=20)
plt.show()

**Análise:** Os histogramas nos permitem observar a forma da distribuição de cada atributo. Por exemplo, `age` (idade) e `chol` (colesterol) parecem ter uma distribuição próxima da normal, enquanto outras variáveis como `cp` (tipo de dor no peito) são claramente categóricas.

#### Balanceamento de Classes

**Objetivo:** Verificar se o número de pacientes com e sem doença cardíaca está equilibrado. Um desbalanceamento severo pode enviesar os modelos de classificação.

In [ ]:
plt.rcParams['figure.figsize'] = (8, 5)
sns.countplot(x=y_binary['num'])
plt.title('Distribuição da Classe Alvo (Doença Cardíaca)', size=16)
plt.xlabel('Presença de Doença Cardíaca (1 = Sim, 0 = Não)')
plt.ylabel('Contagem de Pacientes')
plt.show()

**Análise:** O gráfico mostra que as classes estão razoavelmente balanceadas, com um número ligeiramente maior de pacientes sem a doença (classe 0). Esse balanceamento é bom e não exige técnicas especiais de reamostragem.

#### Matriz de Correlação

**Objetivo:** Identificar quais atributos estão mais fortemente correlacionados entre si e, principalmente, com a variável alvo (`target`). Isso pode indicar quais features são mais preditivas.

In [ ]:
df_corr = X_imputed.copy()
df_corr['target'] = y_binary

plt.figure(figsize=(14, 12))
sns.heatmap(df_corr.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de Correlação entre Atributos e a Classe Alvo', size=16)
plt.show()

**Análise:** O heatmap mostra que `cp` (tipo de dor no peito), `thalach` (frequência cardíaca máxima) e `slope` (inclinação do segmento ST) têm uma correlação positiva moderada com a presença de doença cardíaca. Por outro lado, `exang` (angina induzida por exercício) e `oldpeak` (depressão de ST) têm correlação negativa. Essas variáveis provavelmente serão importantes para os modelos.

## 4. Modelagem de Classificação

**Objetivo:** Implementar e avaliar três algoritmos de classificação para prever a presença de doença cardíaca. As etapas são:
1.  Preparar os dados (escalonamento e divisão treino/teste).
2.  Criar uma função para avaliar os modelos de forma padronizada.
3.  Treinar e avaliar cada um dos três algoritmos.
4.  Comparar os resultados visualmente com a curva ROC.

### 4.1. Preparação dos Dados (Escalonamento e Divisão)

**Justificativa:** Algoritmos como SVM e KNN são sensíveis à escala das variáveis. Por isso, aplicamos o `StandardScaler` para normalizar os dados (média 0 e desvio padrão 1). Em seguida, dividimos os dados em conjuntos de treino (80%) e teste (20%) para avaliar a capacidade de generalização dos modelos.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score, roc_auc_score, roc_curve, silhouette_score
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans, DBSCAN

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_binary, test_size=0.2, random_state=42, stratify=y_binary)
y_train = y_train.values.ravel()

### 4.2. Função de Avaliação

**Objetivo:** Para manter o código limpo e padronizar a avaliação, esta função calcula e exibe todas as métricas exigidas no trabalho: Acurácia, Sensibilidade (Recall), Especificidade, AUC, Matriz de Confusão e a acurácia da validação cruzada (k-fold).

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp)
    
    print(f'Acurácia: {acc:.4f}')
    print(f'Sensibilidade (Recall): {recall:.4f}')
    print(f'Especificidade: {specificity:.4f}')
    print(f'AUC: {auc:.4f}')
    
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='accuracy')
    print(f'Acurácia (Validação Cruzada k=10): {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f}')
    
    plt.figure(figsize=(6, 4))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Matriz de Confusão')
    plt.xlabel('Previsto')
    plt.ylabel('Verdadeiro')
    plt.show()
    return model

### 4.3. Algoritmo 1: Árvore de Decisão

**Justificativa:** Árvores de Decisão são modelos fáceis de interpretar e visualizar. O parâmetro `max_depth=5` foi escolhido para limitar a profundidade da árvore e evitar o superajuste (overfitting) aos dados de treino.

In [ ]:
print("--- Avaliação: Árvore de Decisão ---")
dt_model = DecisionTreeClassifier(random_state=42, max_depth=5)
dt_model = evaluate_model(dt_model, X_train, y_train, X_test, y_test)

### 4.4. Algoritmo 2: K-Nearest Neighbors (KNN)

**Justificativa:** KNN é um algoritmo simples baseado em instâncias que classifica uma amostra com base na classe de seus vizinhos mais próximos. O parâmetro `n_neighbors=7` (número de vizinhos) é um valor comum que oferece um bom equilíbrio entre viés e variância.

In [ ]:
print("\n--- Avaliação: K-Nearest Neighbors (KNN) ---")
knn_model = KNeighborsClassifier(n_neighbors=7)
knn_model = evaluate_model(knn_model, X_train, y_train, X_test, y_test)

### 4.5. Algoritmo 3: Support Vector Machine (SVM)

**Justificativa:** SVM é um modelo poderoso que busca encontrar o hiperplano que melhor separa as classes. O `kernel='rbf'` (Radial Basis Function) foi escolhido por sua capacidade de lidar com relações não lineares nos dados. O parâmetro `probability=True` é necessário para calcular a curva ROC.

In [ ]:
print("\n--- Avaliação: Support Vector Machine (SVM) ---")
svm_model = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
svm_model = evaluate_model(svm_model, X_train, y_train, X_test, y_test)

### 4.6. Comparação dos Modelos (Curva ROC)

**Objetivo:** A curva ROC (Receiver Operating Characteristic) é uma ferramenta visual para comparar o desempenho dos classificadores. Um modelo é considerado melhor quanto mais sua curva se aproxima do canto superior esquerdo, e a área sob a curva (AUC) quantifica esse desempenho.

In [ ]:
plt.figure(figsize=(10, 8))
models = {'Árvore de Decisão': dt_model, 'KNN': knn_model, 'SVM': svm_model}
for name, model in models.items():
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc = roc_auc_score(y_test, y_pred_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('Taxa de Falsos Positivos')
plt.ylabel('Taxa de Verdadeiros Positivos')
plt.title('Curva ROC dos Modelos de Classificação', size=16)
plt.legend()
plt.show()

**Análise:** O SVM apresentou a maior AUC, indicando que é o modelo com o melhor poder de discriminação entre as classes neste dataset.

## 5. Modelagem de Clusterização

**Objetivo:** Aplicar algoritmos de agrupamento para encontrar padrões e segmentar os pacientes em grupos com características similares. Utilizaremos os dados já escalados (`X_scaled`) para que todas as features tenham a mesma importância.

### 5.1. Algoritmo 1: K-Means

**Justificativa:** K-Means é um dos algoritmos de clusterização mais populares devido à sua simplicidade e eficiência. O principal parâmetro a ser definido é o número de clusters (`k`). Para escolhê-lo de forma objetiva, usamos o **Silhouette Score**, uma métrica que avalia quão bem definidas são as separações entre os clusters. O valor de `k` que maximiza essa pontuação é geralmente a melhor escolha.

In [ ]:
silhouette_scores = []
k_range = range(2, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    score = silhouette_score(X_scaled, kmeans.labels_)
    silhouette_scores.append(score)

plt.figure(figsize=(10, 6))
plt.plot(k_range, silhouette_scores, marker='o')
plt.xlabel('Número de Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Análise do Silhouette Score para K-Means', size=16)
plt.show()

best_k = k_range[np.argmax(silhouette_scores)]
print(f'O melhor número de clusters (k) com base no Silhouette Score é: {best_k}')

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

### 5.2. Algoritmo 2: DBSCAN

**Justificativa:** DBSCAN (Density-Based Spatial Clustering of Applications with Noise) é um algoritmo baseado em densidade, útil por não exigir a definição prévia do número de clusters e por ser capaz de identificar pontos como ruído (outliers).

**Parâmetros:**
- `eps=2.5`: Define o raio da vizinhança para um ponto ser considerado "denso".
- `min_samples=5`: O número mínimo de pontos dentro do raio `eps` para formar um cluster.

In [ ]:
dbscan = DBSCAN(eps=2.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)
n_clusters_ = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise_ = list(dbscan_labels).count(-1)
print(f'Número estimado de clusters pelo DBSCAN: {n_clusters_}')
print(f'Número estimado de pontos de ruído: {n_noise_}')

### 5.3. Visualização dos Clusters

**Objetivo:** Como não podemos visualizar 13 dimensões, usamos a **Análise de Componentes Principais (PCA)** para reduzir a dimensionalidade para 2D. Isso nos permite plotar os dados em um gráfico de dispersão e colorir cada ponto de acordo com o cluster atribuído por cada algoritmo.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(18, 8))

plt.subplot(1, 2, 1)
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=kmeans_labels, palette='viridis', s=100, alpha=0.7)
plt.title(f'Clusters K-Means (k={best_k})', size=16)
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend(title='Cluster')

plt.subplot(1, 2, 2)
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=dbscan_labels, palette='viridis', s=100, alpha=0.7)
plt.title('Clusters DBSCAN', size=16)
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend(title='Cluster')

plt.suptitle('Visualização de Clusters com PCA', size=20)
plt.show()

**Análise:** A visualização mostra como cada algoritmo agrupou os dados. O K-Means criou grupos bem definidos, enquanto o DBSCAN identificou um grande cluster principal e classificou vários pontos como ruído (cluster -1), o que pode ser útil para detectar pacientes com perfis atípicos.